# 2.3 核心编程模型

## 概述

上一节我们分清了 Host 与 kernel 的职责。进入 kernel 后，输入数据不会直接在 GM 中完成计算，而是先搬到 UB，再加载到向量寄存器中处理；计算结果随后写回 UB，最后保存到 GM。本节沿着这条数据通路，认识 GM、UB 和寄存器分别存放什么，并学习怎样组织一次简单的搬运与计算。

## 学习目标

- 按位置描述 GM 中的完整 Tensor、UB 中的局部数据和寄存器中的一组数据。
- 使用 `T.alloc_shared` 与 `T.copy` 组织区域搬运。
- 用 `T.simd.vld/vadd/vsts` 完成一维、二维和广播加法，并比较完整输出。

## 1. 内存位置

先用一维 Add 看一次完整的数据流动：输入 A、B 从哪里来，在哪里完成相加，结果 C 又写回哪里。下图上半部分画出完整路径，下半部分展开寄存器每次处理 64 个 `float32` 元素的过程，便于对照后面的循环代码。

<img alt="一维 Add 从 GM 经 UB 和寄存器再写回的两轮计算" src="images/02.03_memory_model_v2.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

图注：以 `N=128` 的一维 Add 为例，A、B 先由 GM 搬入各自的 UB 工作区；两轮向量计算分别处理索引 `[0,64)` 和 `[64,128)`，结果汇入 `c_ub` 后写回 GM 中的 C。图中格内的数字表示**元素索引**，不代表 A、B 或 C 的数值。

## 2. 数据怎样搬到 UB 并完成计算

对照上面的数据流图，kernel 先在 UB 中准备局部空间，将当前数据从 GM 搬进来，再进行向量计算，最后把结果写回 GM。接下来的三个例子沿用这条数据通路，主要使用这些写法：

- `T.alloc_shared(shape, dtype)`：申请 UB 工作区。
- `T.copy(src, dst)`：在本节示例中，把输入从 GM 搬到 UB，再把结果从 UB 写回 GM。
- `with T.SimdVF():`：标出使用向量寄存器计算的代码区域。
- `T.serial(...)`：循环处理多组数据。
- `T.simd.vld/vadd/vsts`：从 UB 读取一组数据、完成向量加法、再写回 UB。

### 先跑一维向量加法

先让一个逻辑块计算 `c[i] = a[i] + b[i]`。输入长度为 128，代码每次处理 64 个 float32 元素，因此要循环两次。编译入口继续沿用 2.2 学过的 `@tilelang.jit`，这里重点看 kernel 内的数据通路。

可以先原样跑通，再把 `run()` 中的 `n` 从 128 改成 256，观察循环处理的数据组数如何变化。**本例要求 `n` 是 64 的正整数倍。**

<p><small>运行顺序：先运行 %%writefile 单元保存脚本，再运行后面的执行单元，比较完整输出。修改代码后也按此顺序重跑。</small></p>

<small>线上没有950设备时，可直接阅读本页结果；自行运行请按1.4准备环境，并从下方准备单元开始顺序执行。辅助代码仅负责启动程序与保存日志。</small>

In [1]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile


def run_example(script):
    """在独立进程中运行，并保留完整日志。"""
    result = subprocess.run([sys.executable, script], capture_output=True, text=True)
    log_dir = Path(os.environ.get("COURSE_OUTPUT_DIR", tempfile.gettempdir())) / "notebook_logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_dir.joinpath(Path(script).stem + ".log").write_text(
        result.stdout + result.stderr, encoding="utf-8"
    )
    # 页内显示程序结果；安装路径、编译过程和底层警告保留在原始日志中。
    print("\n".join(
        line for line in result.stdout.splitlines()
        if not line.startswith("Loading tilelang libs from dev root:")
        and not re.match(r"^\d{4}-\d{2}-\d{2}.*\[TileLang:", line)
    ))
    if result.returncode:
        print(result.stderr)
        result.check_returncode()


In [2]:
%%writefile /tmp/tilelang_ch02_1d_add.py
import torch
import tilelang
import tilelang.language as T


@tilelang.jit(out_idx=-1, target="ascend")
def make_1d_add(n: int = 128):
    # 当前示例每次处理 64 个 float32 元素，先检查输入是否合适。
    if n <= 0 or n % 64:
        raise ValueError("n 必须是 64 的正整数倍")

    @T.prim_func
    def main(
        a: T.Tensor((n,), T.float32),
        b: T.Tensor((n,), T.float32),
        c: T.Tensor((n,), T.float32),
    ):
        with T.Kernel(1):  # 一份逻辑任务处理整个一维 Tensor。
            a_ub = T.alloc_shared((n,), T.float32)
            b_ub = T.alloc_shared((n,), T.float32)
            c_ub = T.alloc_shared((n,), T.float32)

            # CopyIn：把输入从 GM 搬到 UB。
            T.copy(a[:], a_ub)
            T.copy(b[:], b_ub)

            # Compute：在向量计算区域中，每轮处理连续的 64 个元素。
            with T.SimdVF():
                mask = T.simd.pset(32)
                for k in T.serial(n // 64):
                    a_vec = T.simd.vld(a_ub[k * 64])
                    b_vec = T.simd.vld(b_ub[k * 64])
                    c_vec = T.simd.vadd(a_vec, b_vec, mask)
                    T.simd.vsts(c_ub[k * 64], c_vec, mask)

            # CopyOut：把结果从 UB 写回 GM。
            T.copy(c_ub, c[:])

    return main


def run():
    n = 128
    torch.manual_seed(23)
    a = torch.randn(n, device="npu", dtype=torch.float32)
    b = torch.randn(n, device="npu", dtype=torch.float32)

    kernel = make_1d_add(n)
    actual = kernel(a, b)
    torch.npu.synchronize()
    torch.testing.assert_close(actual, a + b, rtol=0.0, atol=0.0)
    print(f"一维加法：{n} 个元素全部通过比较")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_1d_add.py


### 对照代码理解向量计算

代码中的 `with T.SimdVF():`标记向量寄存器计算区域。SIMD（**Single Instruction, Multiple Data**） 指一条指令对一组元素执行相同的操作。在该计算区域内：`vld` 从 UB 读入一组数据，`vadd` 计算，`vsts` 把结果写回 UB。

`T.simd.pset(32)` 用来指定一组元素中哪些位置参与计算和写回，可以把它想成一组开关。

**`32` 表示每个 `float32` 元素占 32 位，不是一次只处理 32 个元素。**

当前使用的向量寄存器宽度为 2048 位，因此一组可容纳 `2048 ÷ 32 = 64` 个 `float32` 元素。不填写其他参数时，`pset(32)` 会把这 64 个位置全部打开。

`vadd` 和 `vsts` 默认处理全部元素，所以即使不传 `mask`，后端也会默认使用全开的掩码；这里显式写出，是为了看清掩码传给了哪些操作。



**`vld` 不接收 `mask`。**

在本例的普通加载模式下，它根据数据类型，从 UB 一次读取完整的 64 个 `float32` 元素。遇到最后一组不足 64 个元素的尾块时，掩码可以限制 `vadd` 的计算位置和 `vsts` 的写回位置，却不能让 `vld` 少读，所以尾块情况还必须保证读取本身安全。

再对照外层代码：`T.Kernel(1)` 表示一份逻辑任务；前后的 `T.copy` 负责 GM 与 UB 之间的搬运；`T.SimdVF` 只标出这份任务中的向量计算区域。故在这段区域里直接读写 GM，是错误的， GM 与 UB 之间的 `T.copy` 要写在区域外。



In [3]:
from pathlib import Path
import subprocess
import sys

# 先运行上方的写入单元，再由独立子进程执行并校验。
script = Path("/tmp/tilelang_ch02_1d_add.py")
if not script.is_file():
    raise FileNotFoundError("请先运行上方代码单元，生成一维加法脚本")
run_example(str(script))


一维加法：128 个元素全部通过比较


### 再看二维矩阵加法

把输入改成两个 `(2, 128)` 矩阵，计算公式仍是 `c[i, j] = a[i, j] + b[i, j]`。向量指令和数据通路都不变，只是在地址中增加行号 `i`：先选一行，再处理该行的两组 64 个元素。下面两格沿用“修改并保存脚本 → 执行并比较”的练习方式。

In [4]:
%%writefile /tmp/tilelang_ch02_2d_add.py
import torch
import tilelang
import tilelang.language as T


@tilelang.jit(out_idx=-1, target="ascend")
def make_2d_add(m: int = 2, n: int = 128):
    # 每行都按 64 个 float32 元素一组处理。
    if m <= 0 or n <= 0 or n % 64:
        raise ValueError("m 必须为正数，n 必须是 64 的正整数倍")

    @T.prim_func
    def main(
        a: T.Tensor((m, n), T.float32),
        b: T.Tensor((m, n), T.float32),
        c: T.Tensor((m, n), T.float32),
    ):
        with T.Kernel(1):
            # UB 工作区与输入矩阵具有相同的二维形状。
            a_ub = T.alloc_shared((m, n), T.float32)
            b_ub = T.alloc_shared((m, n), T.float32)
            c_ub = T.alloc_shared((m, n), T.float32)

            # CopyIn：把两块完整矩阵搬入 UB。
            T.copy(a[0:m, 0:n], a_ub)
            T.copy(b[0:m, 0:n], b_ub)

            with T.SimdVF():
                mask = T.simd.pset(32)
                for i in T.serial(m):  # 先选第 i 行。
                    for k in T.serial(n // 64):  # 再选该行中的第 k 组。
                        a_vec = T.simd.vld(a_ub[i, k * 64])
                        b_vec = T.simd.vld(b_ub[i, k * 64])
                        c_vec = T.simd.vadd(a_vec, b_vec, mask)
                        T.simd.vsts(c_ub[i, k * 64], c_vec, mask)

            # CopyOut：二维结果写回 C 的对应位置。
            T.copy(c_ub, c[0:m, 0:n])

    return main


def run():
    m, n = 2, 128
    torch.manual_seed(23)
    a = torch.randn(m, n, device="npu", dtype=torch.float32)
    b = torch.randn(m, n, device="npu", dtype=torch.float32)

    kernel = make_2d_add(m, n)
    actual = kernel(a, b)
    torch.npu.synchronize()
    torch.testing.assert_close(actual, a + b, rtol=0.0, atol=0.0)
    print(f"二维加法：{m} 行、每行 {n} 个元素全部通过比较")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_2d_add.py


In [5]:
from pathlib import Path
import subprocess
import sys

# 先运行上方的写入单元，再由独立子进程执行并校验。
script = Path("/tmp/tilelang_ch02_2d_add.py")
if not script.is_file():
    raise FileNotFoundError("请先运行上方代码单元，生成二维加法脚本")
run_example(str(script))


二维加法：2 行、每行 128 个元素全部通过比较


这次 `a_ub[i, k * 64]` 比一维写法多了行号 `i`。`i` 依次选两行，`k` 在每行中依次选前 64 个、后 64 个元素；`T.simd.vadd` 本身没有变化。

### 最后看广播：给每一行加同一个向量

这次 `a` 是 `(2, 128)` 矩阵，`b` 是 `(128,)` 向量，计算 `c[i, j] = a[i, j] + b[j]`。同一段 `b` 会用于矩阵的每一行；代码先读取 `b` 的一组元素，再依次与两行对应位置相加。你可以继续在第一格修改代码，再用第二格检查完整输出。

In [6]:
%%writefile /tmp/tilelang_ch02_broadcast_add.py
import torch
import tilelang
import tilelang.language as T


@tilelang.jit(out_idx=-1, target="ascend")
def make_broadcast_add(m: int = 2, n: int = 128):
    # 矩阵每行和向量都按 64 个 float32 元素一组处理。
    if m <= 0 or n <= 0 or n % 64:
        raise ValueError("m 必须为正数，n 必须是 64 的正整数倍")

    @T.prim_func
    def main(
        a: T.Tensor((m, n), T.float32),  # 二维矩阵。
        b: T.Tensor((n,), T.float32),    # 所有行共用的一维向量。
        c: T.Tensor((m, n), T.float32),
    ):
        with T.Kernel(1):
            a_ub = T.alloc_shared((m, n), T.float32)
            b_ub = T.alloc_shared((n,), T.float32)
            c_ub = T.alloc_shared((m, n), T.float32)

            # CopyIn：a 是矩阵，b 是只需搬入一次的向量。
            T.copy(a[0:m, 0:n], a_ub)
            T.copy(b[:], b_ub)

            with T.SimdVF():
                mask = T.simd.pset(32)
                for k in T.serial(n // 64):
                    # 先读取 b 的一组数据，然后让每一行重复使用。
                    b_vec = T.simd.vld(b_ub[k * 64])
                    for i in T.serial(m):
                        a_vec = T.simd.vld(a_ub[i, k * 64])
                        c_vec = T.simd.vadd(a_vec, b_vec, mask)
                        T.simd.vsts(c_ub[i, k * 64], c_vec, mask)

            T.copy(c_ub, c[0:m, 0:n])

    return main


def run():
    m, n = 2, 128
    torch.manual_seed(23)
    a = torch.randn(m, n, device="npu", dtype=torch.float32)
    b = torch.randn(n, device="npu", dtype=torch.float32)

    kernel = make_broadcast_add(m, n)
    actual = kernel(a, b)
    torch.npu.synchronize()
    # PyTorch 把 b 加到 a 的每一行；比较矩阵中的每个元素。
    torch.testing.assert_close(actual, a + b, rtol=0.0, atol=0.0)
    print(f"广播加法：一维向量加到矩阵的 {m} 行、每行 {n} 个元素全部通过比较")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_broadcast_add.py


In [7]:
from pathlib import Path
import subprocess
import sys

# 先运行上方的写入单元，再由独立子进程执行并校验。
script = Path("/tmp/tilelang_ch02_broadcast_add.py")
if not script.is_file():
    raise FileNotFoundError("请先运行上方代码单元，生成广播加法脚本")
run_example(str(script))


广播加法：一维向量加到矩阵的 2 行、每行 128 个元素全部通过比较


对比二维加法，这里只有 `b` 的形状和读取位置变了：`b_ub[k * 64]` 没有行号，可以在处理不同的行时重复使用。三个例子都经过完整输出比较；它们共享同一条 **GM → UB → 寄存器 → UB → GM** 数据通路。

## 3. 换一种计算：ReLU

前面的三个例子都在做加法。接下来把计算改成 ReLU，看数据搬运与写回仍按同一条路径进行。ReLU 的公式是：

```text
y[i] = max(x[i], 0)
```

它只有一个输入和一个输出，数据通路仍分成三步：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">步骤</th>
<th style="text-align: left; vertical-align: top;">主要代码</th>
<th style="text-align: left; vertical-align: top;">数据怎样移动</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">CopyIn</td>
<td style="text-align: left; vertical-align: top;"><code>T.copy(A[:], x_ub)</code></td>
<td style="text-align: left; vertical-align: top;">输入从 GM 搬到 UB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">Compute</td>
<td style="text-align: left; vertical-align: top;"><code>vld → vmax → vsts</code></td>
<td style="text-align: left; vertical-align: top;">数据进入寄存器计算，再写回 UB</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">CopyOut</td>
<td style="text-align: left; vertical-align: top;"><code>T.copy(y_ub, C[:])</code></td>
<td style="text-align: left; vertical-align: top;">结果从 UB 写回 GM</td>
</tr>
</tbody>
</table>

这次请自己把数据通路写完整。第一格保留了算子接口、Host 输入和结果校验；`T.Kernel(1)` 内只留下四步注释，需要你补上 **UB 工作区、搬入、计算和写回**。

检查完整输出是否与 `torch.relu` 一致。写完遇到问题时可对照 [ReLU 参考实现](answer/02.03_relu_model.py)。

<p><small>补全后，先运行写入单元保存脚本，再运行后面的执行单元。</small></p>

In [8]:
%%writefile /tmp/tilelang_ch02_relu_practice.py
import torch
import tilelang
import tilelang.language as T


@tilelang.jit(out_idx=-1, target="ascend")
def make_relu(n: int = 1024):
    # 当前例子只处理完整的 64 元素组。
    if n <= 0 or n % 64:
        raise ValueError("n 必须是 64 的正整数倍")

    @T.prim_func
    def main(A: T.Tensor((n,), T.float32), C: T.Tensor((n,), T.float32)):
        with T.Kernel(1):
            # 1. 在 UB 中申请两个长度为 n 的 float32 工作区：x_ub 和 y_ub。
            # 2. CopyIn：把 GM 中的 A 搬入 x_ub。
            # 3. Compute：进入 SimdVF，以 64 个元素为一组：
            #    从 x_ub 读入寄存器，计算 max(x, 0)，再写入 y_ub。
            # 4. CopyOut：把 y_ub 中的结果写回 GM 中的 C。
            pass  # 写好以上四步后，可以删掉这一行。

    return main


def run():
    n = 1024
    torch.manual_seed(23)
    x = torch.randn(n, device="npu", dtype=torch.float32)

    kernel = make_relu(n)
    actual = kernel(x)
    torch.npu.synchronize()
    torch.testing.assert_close(actual, torch.relu(x), rtol=0.0, atol=0.0)
    print(f"ReLU：{n} 个元素全部通过比较")


if __name__ == "__main__":
    run()


Overwriting /tmp/tilelang_ch02_relu_practice.py


In [9]:
from pathlib import Path
import subprocess
import sys

# 修改上方代码后，先重跑写入单元，再运行本单元。
script = Path("/tmp/tilelang_ch02_relu_practice.py")
if not script.is_file():
    print("请先运行上方代码单元，生成 ReLU 练习脚本。")
elif "pass  # 写好以上四步后，可以删掉这一行。" in script.read_text(encoding="utf-8"):
    print("ReLU 尚未执行：请补全四步代码并删除占位 pass，再重新运行两个单元。")
else:
    run_example(str(script))

ReLU 尚未执行：请补全四步代码并删除占位 pass，再重新运行两个单元。


### 参考答案的运行结果

下面执行[完整ReLU参考答案](answer/02.03_relu_model.py)，比较全部1024个输出元素。**这份结果来自参考答案，不代表上面的填空练习已经完成。**

In [10]:
import subprocess, sys

run_example("answer/02.03_relu_model.py")

ReLU：1024 个元素全部通过比较


## 小结

本节通过一维加法、二维加法和广播加法，看到了同一条 GM → UB → 寄存器 → UB → GM 数据通路怎样适应不同输入形状；ReLU 则说明计算公式改变时，仍可沿着这条路径检查数据是否正确搬入、计算和写回。下一节将在 Add 的基础上明确算子规格，把数据分给两个逻辑块，并对照 JIT 与显式编译两种入口。

## 课后练习

1. 根据上面的代码骨架实现 ReLU；依次运行写入单元和执行单元，确认完整输出与 `torch.relu` 一致。
2. 按照 `A → x_ub → x_vec → y_vec → y_ub → C`，说明每一步数据位于 GM、UB 还是寄存器，以及两次 `T.copy` 的方向。
3. 当 `n=1024`、dtype 为 float32 时，计算 `x_ub` 和 `y_ub` 一共占用多少字节。

答案与解析见 [02.03_questions.txt](answer/02.03_questions.txt)。